# Task 2: Heart Disease Prediction System

**Name:** Muhammad Hassan Yousaf  
**Roll Number:** FA24-BSE-082  
**Course:** Machine Learning (CSC-461)  

### Live Cloud Deployment
- **Streamlit App:** https://heart-disease-model-hassan.streamlit.app
- **Modal App:** https://mhassan0000--heart-disease-predictor-run.modal.run
- **GitHub Repository:** https://github.com/MHassan0000/ML-Prediction-Models

---
### Assignment Constraint
As specified in the assignment instructions, we select **only 4 input attributes** from the UCI Heart Disease dataset:
1. `age`: Patient age in years
2. `thalach`: Maximum heart rate achieved during exercise stress test
3. `chol`: Serum cholesterol in mg/dl
4. `cp`: Chest pain type (0 to 3)

We apply the exact same ML lifecycle steps as Task 1.



## 0. Import Libraries
Import libraries for analysis, visualization, and modeling.


In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


## 1. Data Collection
Load the UCI Heart Disease dataset (`heart.csv`).


In [ ]:
# Load heart disease dataset
data_path = "data/heart.csv" if os.path.exists("data/heart.csv") else "heart.csv"

if os.path.exists(data_path):
    df_heart = pd.read_csv(data_path)
    print(f"Loaded dataset from {data_path}")
else:
    raise FileNotFoundError("heart.csv not found in data/ or root")

print("Dataset shape:", df_heart.shape)
df_heart.head()


## 2. Data Understanding and Exploratory Data Analysis
Inspect dataset summary, target balance, and feature distributions.


In [ ]:
# Dataset info and statistics
print("Dataset Info:")
df_heart.info()

print("\nDescriptive Statistics:")
df_heart.describe()


In [ ]:
# Target distribution (0 = No Disease, 1 = Disease)
print("Target counts:")
print(df_heart['target'].value_counts())


In [ ]:
# EDA plots for the 4 attributes
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Target count
sns.countplot(data=df_heart, x='target', ax=axes[0, 0], palette='Reds')
axes[0, 0].set_title('Heart Disease Target Distribution')

# 2. Chest pain type vs target
sns.countplot(data=df_heart, x='cp', hue='target', ax=axes[0, 1], palette='coolwarm')
axes[0, 1].set_title('Heart Disease by Chest Pain Type')

# 3. Max heart rate boxplot
sns.boxplot(data=df_heart, x='target', y='thalach', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Max Heart Rate vs Target')

# 4. Cholesterol vs Age
sns.scatterplot(data=df_heart, x='age', y='chol', hue='target', ax=axes[1, 1], palette='magma')
axes[1, 1].set_title('Age vs Cholesterol')

plt.tight_layout()
os.makedirs("reports", exist_ok=True)
plt.savefig("reports/heart_disease_eda.png", dpi=100)
plt.show()


In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df_heart.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title("Heart Disease Feature Correlation")
plt.tight_layout()
plt.savefig("reports/heart_disease_correlation.png", dpi=100)
plt.show()


## 3. Data Preprocessing and 4-Attribute Selection
Isolate the 4 required attributes: `age`, `thalach`, `chol`, and `cp`. Encode `cp` and verify missing values.


In [ ]:
# Select the 4 mandated features
selected_features = ['age', 'thalach', 'chol', 'cp']
target_col = 'target'

df_sub = df_heart[selected_features + [target_col]].dropna().copy()
print("Selected dataset shape:", df_sub.shape)

# Encode chest pain type (cp)
le_cp = LabelEncoder()
df_sub['cp'] = le_cp.fit_transform(df_sub['cp'])

# Save encoded dataset
os.makedirs("data", exist_ok=True)
df_sub.to_csv("data/heart_disease_encoded.csv", index=False)
df_sub.head()


## 4. Model Training (SVC with RBF Kernel)
Split the data into train (80%) and test (20%) sets, then train the Support Vector Classifier.


In [ ]:
# Train-test split
X = df_sub[selected_features]
y = df_sub[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Train SVC
svc_heart = SVC(kernel='rbf', random_state=42)
svc_heart.fit(X_train, y_train)
print("SVC model trained successfully.")


## 5. Model Evaluation
Evaluate performance using accuracy score, confusion matrix, and classification report.


In [ ]:
# Evaluate model
y_pred = svc_heart.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Test Accuracy: {acc * 100:.2f}%\n")
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

# Plot confusion matrix
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title(f"Confusion Matrix (Accuracy: {acc*100:.1f}%)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("reports/heart_disease_confusion_matrix.png", dpi=100)
plt.show()


## 6. Testing Predictions
Run predictions on sample patient data with different clinical attributes.


In [ ]:
# Sample patient testing
patients = pd.DataFrame([
    {"age": 52, "thalach": 168, "chol": 212, "cp": 0},
    {"age": 67, "thalach": 108, "chol": 300, "cp": 1},
    {"age": 45, "thalach": 150, "chol": 225, "cp": 2},
    {"age": 38, "thalach": 170, "chol": 174, "cp": 3},
    {"age": 60, "thalach": 120, "chol": 280, "cp": 0}
])

patients_enc = patients.copy()
patients_enc['cp'] = le_cp.transform(patients_enc['cp'])

preds = svc_heart.predict(patients_enc[selected_features])
patients['Prediction'] = ['Heart Disease' if p == 1 else 'No Heart Disease' for p in preds]

os.makedirs("reports", exist_ok=True)
patients.to_csv("reports/heart_disease_predictions.csv", index=False)
patients


## 7. Saving the Trained Model
Save the trained SVC model and chest pain label encoder.


In [ ]:
# Save model and label encoder
os.makedirs("models", exist_ok=True)

with open("models/heart_disease_svc_model.pkl", "wb") as f:
    pickle.dump(svc_heart, f)

with open("models/heart_disease_le_cp.pkl", "wb") as f:
    pickle.dump(le_cp, f)

print("Saved model to models/heart_disease_svc_model.pkl")
print("Saved encoder to models/heart_disease_le_cp.pkl")
